In [1]:
import pandas as pd
from pathlib import Path

DATA = Path(r"C:\Projets\VoicePrint_Piscine\data\raw\cv26_fr\fr-FR")
print("Le dossier existe :", DATA.exists())

cols = ["client_id", "path", "age", "gender"]

# quoting=3 : ignore les guillemets, qui posent problème dans ces fichiers
train = pd.read_csv(DATA / "train.tsv", sep="\t", usecols=cols, quoting=3)
dev   = pd.read_csv(DATA / "dev.tsv",   sep="\t", usecols=cols, quoting=3)
test  = pd.read_csv(DATA / "test.tsv",  sep="\t", usecols=cols, quoting=3)
print(len(train), len(dev), len(test))

Le dossier existe : True
5086 271 133


In [2]:
tout = pd.concat([train.assign(split="train"), dev.assign(split="dev"), test.assign(split="test")])
print("Locuteurs (identifiants uniques) :", tout["client_id"].nunique())

clips_par_locuteur = tout.groupby("client_id").size()
print(clips_par_locuteur.describe())

for n in [2, 5, 10, 20]:
    print(f"Locuteurs avec au moins {n} clips :", (clips_par_locuteur >= n).sum())

Locuteurs (identifiants uniques) : 206
count    206.000000
mean      26.650485
std       65.411300
min        1.000000
25%        4.000000
50%        5.000000
75%       15.000000
max      505.000000
dtype: float64
Locuteurs avec au moins 2 clips : 201
Locuteurs avec au moins 5 clips : 142
Locuteurs avec au moins 10 clips : 64
Locuteurs avec au moins 20 clips : 39


In [3]:
ids = {s: set(tout.loc[tout["split"] == s, "client_id"]) for s in ["train", "dev", "test"]}
print("Locuteurs par groupe :", {s: len(v) for s, v in ids.items()})
print("train et dev en commun  :", len(ids["train"] & ids["dev"]))
print("train et test en commun :", len(ids["train"] & ids["test"]))
print("dev et test en commun   :", len(ids["dev"] & ids["test"]))

Locuteurs par groupe : {'train': 106, 'dev': 57, 'test': 43}
train et dev en commun  : 0
train et test en commun : 0
dev et test en commun   : 0


In [4]:
garde = clips_par_locuteur[clips_par_locuteur >= 5]
print("Locuteurs gardés :", len(garde))
print("Clips correspondants :", garde.sum(), "sur", len(tout))
top5 = clips_par_locuteur.nlargest(5)
print("Les 5 plus gros contributeurs (nombre de clips) :", top5.tolist())
print("Leur part du total :", round(top5.sum() / len(tout) * 100, 1), "%")

Locuteurs gardés : 142
Clips correspondants : 5287 sur 5490
Les 5 plus gros contributeurs (nombre de clips) : [505, 424, 324, 290, 243]
Leur part du total : 32.5 %
